# Block 5: Risk Scoring and Recommendation Engine

This notebook converts model outputs into operational decisions.

Inputs:

- `outputs/cleaned/cleaned_events_with_labels.csv`
- `outputs/model1_v2/model1_v2_duration_band_with_road_closure_features.csv`
- `outputs/model2_v2/model2_v2_duration_band_model.pkl`

Outputs:

- `outputs/recommendations/event_risk_playbooks.csv`
- `outputs/recommendations/hotspot_summary.csv`
- `outputs/recommendations/recommendation_rules.json`
- `outputs/recommendations/recommendation_summary.csv`

The recommendation engine is rule-based because the dataset does not contain ground-truth labels for optimal deployment. The rules use model predictions, hotspot signals, and event cause overrides.

## 1. Imports and Paths

In [1]:
from pathlib import Path
import json
import warnings

import joblib
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 160)
pd.set_option('display.max_rows', 100)

cwd = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [cwd, *cwd.parents] if (p / 'outputs').exists()), cwd)
CLEANED_PATH = PROJECT_ROOT / 'outputs' / 'cleaned' / 'cleaned_events_with_labels.csv'
STACKED_FEATURES_PATH = PROJECT_ROOT / 'outputs' / 'model1_v2' / 'model1_v2_duration_band_with_road_closure_features.csv'
MODEL2_PATH = PROJECT_ROOT / 'outputs' / 'model2_v2' / 'model2_v2_duration_band_model.pkl'
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'recommendations'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Project root:', PROJECT_ROOT)
print('Cleaned data:', CLEANED_PATH)
print('Stacked features:', STACKED_FEATURES_PATH)
print('Model 2 artifact:', MODEL2_PATH)
print('Output dir:', OUTPUT_DIR)

Project root: /Users/astron_designer/GridLock_Phase2
Cleaned data: /Users/astron_designer/GridLock_Phase2/outputs/cleaned/cleaned_events_with_labels.csv
Stacked features: /Users/astron_designer/GridLock_Phase2/outputs/model1_v2/model1_v2_duration_band_with_road_closure_features.csv
Model 2 artifact: /Users/astron_designer/GridLock_Phase2/outputs/model2_v2/model2_v2_duration_band_model.pkl
Output dir: /Users/astron_designer/GridLock_Phase2/outputs/recommendations


## 2. Load Inputs

In [2]:
cleaned = pd.read_csv(CLEANED_PATH, low_memory=False)
stacked = pd.read_csv(STACKED_FEATURES_PATH, low_memory=False)
model2_artifact = joblib.load(MODEL2_PATH)

model2 = model2_artifact['model']
model2_name = model2_artifact.get('model_name', 'unknown')
feature_cols = model2_artifact['feature_cols']
label_encoder = model2_artifact['label_encoder']

valid_context = cleaned.loc[cleaned['valid_duration_label'].eq(1)].reset_index(drop=True).copy()
stacked = stacked.reset_index(drop=True).copy()

assert len(valid_context) == len(stacked), 'Context rows and stacked feature rows must align.'
assert 'road_closure_probability' in stacked.columns, 'Missing Model 1 stacked probability.'

print('Valid context shape:', valid_context.shape)
print('Stacked feature shape:', stacked.shape)
print('Model 2:', model2_name)
print('Model 2 features:', len(feature_cols))
print('Duration classes:', list(label_encoder.classes_))

Valid context shape: (2764, 72)
Stacked feature shape: (2764, 537)
Model 2: RandomForest
Model 2 features: 536
Duration classes: ['long', 'medium', 'short']


## 3. Predict Duration Band for All Valid-Duration Rows

Block 4 evaluated a held-out test split. Here we score every valid-duration row for recommendation/demo output.

In [ ]:
X_all = stacked.drop(columns=['duration_band']).copy()

# Use selected_features if stored in artifact, else fall back to feature_cols
model_feature_cols = model2_artifact.get('selected_features') or feature_cols

missing_cols = [c for c in model_feature_cols if c not in X_all.columns]
extra_cols = [c for c in X_all.columns if c not in model_feature_cols]

for col in missing_cols:
    X_all[col] = 0
X_all = X_all[model_feature_cols]

for col in X_all.columns:
    X_all[col] = pd.to_numeric(X_all[col], errors='coerce')
X_all = X_all.replace([np.inf, -np.inf], np.nan).fillna(0)

duration_pred_encoded = np.asarray(model2.predict(X_all)).astype(int).ravel()
duration_pred = label_encoder.inverse_transform(duration_pred_encoded)

if hasattr(model2, 'predict_proba'):
    duration_proba = model2.predict_proba(X_all)
else:
    duration_proba = np.zeros((len(X_all), len(label_encoder.classes_)))

prob_df = pd.DataFrame(duration_proba, columns=[f'prob_duration_{c}' for c in label_encoder.classes_])

# ── NEW: Prediction confidence (max class probability) ──
prediction_confidence = duration_proba.max(axis=1)

print('Missing feature columns filled:', len(missing_cols))
print('Extra feature columns ignored:', len(extra_cols))
print('Model features used:', len(model_feature_cols))
print('Predicted duration distribution:')
display(pd.Series(duration_pred).value_counts().to_frame('count'))
print(f'\nMean prediction confidence: {prediction_confidence.mean():.3f}')

Missing feature columns filled: 0
Extra feature columns ignored: 0
Predicted duration distribution:


,count
short,1674
medium,770
long,320


## 4. Build Hotspot Summary

Hotspot statistics help the playbook distinguish isolated events from repeat-risk locations.

In [4]:
for col in ['event_cause', 'corridor', 'junction', 'police_station', 'zone']:
    if col in valid_context.columns:
        valid_context[col] = valid_context[col].fillna('unknown').astype(str).str.strip().str.lower().replace({'': 'unknown'})

hotspot_summary = (
    cleaned.assign(
        corridor=cleaned.get('corridor', pd.Series('unknown', index=cleaned.index)).fillna('unknown').astype(str).str.strip().str.lower(),
        junction=cleaned.get('junction', pd.Series('unknown', index=cleaned.index)).fillna('unknown').astype(str).str.strip().str.lower(),
        police_station=cleaned.get('police_station', pd.Series('unknown', index=cleaned.index)).fillna('unknown').astype(str).str.strip().str.lower(),
        event_cause=cleaned.get('event_cause', pd.Series('unknown', index=cleaned.index)).fillna('unknown').astype(str).str.strip().str.lower(),
    )
    .groupby(['corridor', 'junction', 'police_station'], dropna=False)
    .agg(
        total_events=('target_road_closure', 'size'),
        closure_events=('target_road_closure', 'sum'),
        closure_rate=('target_road_closure', 'mean'),
        valid_duration_events=('valid_duration_label', 'sum'),
        long_duration_events=('duration_band', lambda s: (s == 'long').sum()),
        accident_events=('event_cause', lambda s: s.str.contains('accident', na=False).sum()),
        breakdown_events=('event_cause', lambda s: s.str.contains('breakdown', na=False).sum()),
        water_logging_events=('event_cause', lambda s: s.str.contains('water', na=False).sum()),
    )
    .reset_index()
)

hotspot_summary['hotspot_score'] = (
    hotspot_summary['total_events'].clip(upper=100) * 0.35
    + hotspot_summary['closure_rate'].fillna(0) * 100 * 0.45
    + hotspot_summary['long_duration_events'].clip(upper=25) * 0.80
)
hotspot_summary['hotspot_level'] = pd.cut(
    hotspot_summary['hotspot_score'],
    bins=[-0.01, 15, 35, 60, np.inf],
    labels=['low', 'medium', 'high', 'critical']
).astype(str)

hotspot_lookup = hotspot_summary.set_index(['corridor', 'junction', 'police_station'])[['total_events', 'closure_rate', 'hotspot_score', 'hotspot_level']]

print('Hotspot rows:', len(hotspot_summary))
display(hotspot_summary.sort_values('hotspot_score', ascending=False).head(20))

Hotspot rows: 642


,corridor,junction,police_station,total_events,closure_events,closure_rate,valid_duration_events,long_duration_events,accident_events,breakdown_events,water_logging_events,hotspot_score,hotspot_level
519,orr east 2,unknown,hal old airport,137,3,0.021898,97,83,0,26,5,55.985401,high
383,non-corridor,"trilight circle,race course road",high ground,2,2,1.000000,1,0,0,1,0,45.700000,high
293,non-corridor,k r circle,high ground,2,2,1.000000,0,0,0,2,0,45.700000,high
173,mysore road,richmond circle jn,cubbon park,2,2,1.000000,0,0,0,2,0,45.700000,high
439,non-corridor,upparpetjunction,upparpet,2,2,1.000000,0,0,0,0,0,45.700000,high
338,non-corridor,nagatheaterjunc-ulsoor,pulikeshinagar(f.town),1,1,1.000000,0,0,0,0,0,45.350000,high
624,west of chord road,modibridgejunction,rajajinagar,1,1,1.000000,0,0,0,0,1,45.350000,high
119,hosur road,unknown,mico layout,1,1,1.000000,0,0,0,0,0,45.350000,high
135,magadi road,khb junction,kamakshipalya,1,1,1.000000,0,0,0,1,0,45.350000,high
178,mysore road,shivajitalkiesjunc,city market,1,1,1.000000,0,0,0,0,0,45.350000,high


## 5. Recommendation Rules

The score combines closure probability, duration severity, hotspot signal, peak-hour context, and event-cause overrides.

In [ ]:
DURATION_POINTS = {'short': 8, 'medium': 22, 'long': 38}
RISK_THRESHOLDS = {'low': 30, 'medium': 55, 'high': 75}

CAUSE_OVERRIDES = {
    'vehicle_breakdown': {
        'risk_bonus': 8,
        'equipment': ['tow van', 'clearance crew'],
        'agency': ['traffic police tow unit'],
    },
    'accident': {
        'risk_bonus': 14,
        'equipment': ['ambulance standby', 'tow van', 'crash cones'],
        'agency': ['traffic police', 'medical emergency support'],
    },
    'tree_fall': {
        'risk_bonus': 16,
        'equipment': ['barricades', 'tree clearance crew'],
        'agency': ['traffic police', 'BBMP tree clearance'],
    },
    'water_logging': {
        'risk_bonus': 14,
        'equipment': ['barricades', 'water pump support'],
        'agency': ['traffic police', 'BBMP/BWSSB'],
    },
    'public_event': {
        'risk_bonus': 12,
        'equipment': ['barricades', 'crowd-control ropes', 'portable signage'],
        'agency': ['traffic police', 'local law and order unit'],
    },
    'procession': {
        'risk_bonus': 12,
        'equipment': ['barricades', 'route marshals', 'portable signage'],
        'agency': ['traffic police', 'local law and order unit'],
    },
    'vip_movement': {
        'risk_bonus': 18,
        'equipment': ['barricades', 'pilot route signage'],
        'agency': ['traffic police', 'VIP movement coordination cell'],
    },
    'construction': {
        'risk_bonus': 8,
        'equipment': ['barricades', 'reflective cones', 'warning signage'],
        'agency': ['traffic police', 'road works contractor'],
    },
    'congestion': {
        'risk_bonus': 6,
        'equipment': ['traffic cones', 'portable signage'],
        'agency': ['traffic police'],
    },
    'pot_holes': {
        'risk_bonus': 5,
        'equipment': ['reflective cones', 'warning signage'],
        'agency': ['traffic police', 'BBMP road maintenance'],
    },
}

BASE_PLAYBOOK = {
    'low': {
        'manpower': '1 patrol unit',
        'barricading': 'No barricading by default; keep cones ready if lane obstruction grows.',
        'diversion': 'No diversion. Monitor junction approach roads.',
        'control_room': 'Log and monitor.',
    },
    'medium': {
        'manpower': '2 officers + 1 patrol unit',
        'barricading': 'Local channelization with cones/signage near incident point.',
        'diversion': 'Prepare local diversion if queue spills to adjacent junction.',
        'control_room': 'Notify station control and monitor every 15 minutes.',
    },
    'high': {
        'manpower': '4-6 officers + 2 patrol units',
        'barricading': 'Barricade readiness at affected arm and upstream junction.',
        'diversion': 'Activate partial diversion and push advisory to nearby corridors.',
        'control_room': 'Control-room loop with field updates every 10 minutes.',
    },
    'critical': {
        'manpower': '8+ officers + inspector oversight + 3 patrol units',
        'barricading': 'Full barricade deployment and protected emergency lane.',
        'diversion': 'Activate full diversion plan and public advisory immediately.',
        'control_room': 'Dedicated control-room monitoring until clearance.',
    },
}

# ── NEW: Confidence-based scaling ──
CONFIDENCE_WEIGHT = {
    'high': 1.0,    # confidence >= 0.70
    'medium': 0.85, # confidence >= 0.50
    'low': 0.70,    # confidence < 0.50
}

recommendation_rules = {
    'duration_points': DURATION_POINTS,
    'risk_thresholds': RISK_THRESHOLDS,
    'cause_overrides': CAUSE_OVERRIDES,
    'base_playbook': BASE_PLAYBOOK,
    'confidence_weight': CONFIDENCE_WEIGHT,
}

print('Rules loaded. Cause overrides:', len(CAUSE_OVERRIDES))

Rules loaded.


## 6. Scoring Functions

In [ ]:
def normalize_text(value):
    if pd.isna(value):
        return 'unknown'
    text = str(value).strip().lower()
    return text if text else 'unknown'


def identify_cause_key(cause_text):
    cause = normalize_text(cause_text)
    for key in CAUSE_OVERRIDES:
        if key in cause:
            return key
    return 'default'


def risk_level_from_score(score):
    if score >= RISK_THRESHOLDS['high']:
        return 'critical'
    if score >= RISK_THRESHOLDS['medium']:
        return 'high'
    if score >= RISK_THRESHOLDS['low']:
        return 'medium'
    return 'low'


def bump_level(level, steps=1):
    order = ['low', 'medium', 'high', 'critical']
    idx = min(order.index(level) + steps, len(order) - 1)
    return order[idx]


def confidence_bucket(conf):
    if conf >= 0.70:
        return 'high'
    if conf >= 0.50:
        return 'medium'
    return 'low'


def build_recommendation(row):
    closure_probability = float(row['road_closure_probability'])
    duration_band = normalize_text(row['predicted_duration_band'])
    cause_key = identify_cause_key(row.get('event_cause', 'unknown'))
    hotspot_score = float(row.get('hotspot_score', 0) or 0)
    peak_bonus = 5 if int(row.get('is_peak_hour', 0) or 0) == 1 else 0
    blocked_bonus = 4 if int(row.get('has_blocked_word', 0) or 0) == 1 else 0
    cause_bonus = CAUSE_OVERRIDES.get(cause_key, {}).get('risk_bonus', 0)

    # ── NEW: Confidence-weighted duration contribution ──
    dur_confidence = float(row.get('prediction_confidence', 0.5))
    conf_bucket = confidence_bucket(dur_confidence)
    conf_weight = CONFIDENCE_WEIGHT[conf_bucket]
    
    duration_contribution = DURATION_POINTS.get(duration_band, 8) * conf_weight

    score = (
        closure_probability * 45
        + duration_contribution
        + min(hotspot_score, 40) * 0.30
        + peak_bonus
        + blocked_bonus
        + cause_bonus
    )
    score = float(np.clip(score, 0, 100))
    level = risk_level_from_score(score)

    if duration_band == 'long' and closure_probability >= 0.60:
        level = bump_level(level, 1)
    if cause_key in {'accident', 'tree_fall', 'vip_movement'} and closure_probability >= 0.50:
        level = bump_level(level, 1)
    # ── NEW: Downgrade if both models have low confidence ──
    if conf_bucket == 'low' and closure_probability < 0.30 and level in {'high', 'critical'}:
        level = bump_level(level, -1) if level == 'high' else level

    base = BASE_PLAYBOOK[level]
    override = CAUSE_OVERRIDES.get(cause_key, {})
    equipment = sorted(set(['traffic cones', 'reflective jackets'] + override.get('equipment', [])))
    agencies = sorted(set(['traffic police'] + override.get('agency', [])))

    if closure_probability >= 0.70 or level in {'high', 'critical'}:
        equipment = sorted(set(equipment + ['barricades', 'portable signage']))
    if duration_band == 'long':
        agencies = sorted(set(agencies + ['control room']))

    playbook = {
        'risk_level': level,
        'risk_score': round(score, 2),
        'confidence_bucket': conf_bucket,
        'manpower': base['manpower'],
        'barricading': base['barricading'],
        'diversion': base['diversion'],
        'control_room': base['control_room'],
        'equipment': equipment,
        'agency_alerts': agencies,
        'primary_trigger': cause_key,
    }
    return playbook

## 7. Build Event-Level Playbooks

In [ ]:
context_cols = [
    'id', 'event_type', 'event_cause', 'corridor', 'junction', 'police_station', 'zone',
    'latitude', 'longitude', 'start_datetime', 'description', 'target_road_closure',
    'duration_band', 'is_peak_hour', 'has_blocked_word', 'valid_duration_label'
]
context_cols = [c for c in context_cols if c in valid_context.columns]

risk_df = valid_context[context_cols].copy()
risk_df['actual_duration_band'] = risk_df.get('duration_band')
risk_df['predicted_duration_band'] = duration_pred
risk_df['road_closure_probability'] = stacked['road_closure_probability'].values
risk_df['prediction_confidence'] = prediction_confidence
risk_df = pd.concat([risk_df, prob_df], axis=1)

# Attach hotspot stats by location group.
for col in ['corridor', 'junction', 'police_station']:
    if col in risk_df.columns:
        risk_df[col] = risk_df[col].fillna('unknown').astype(str).str.strip().str.lower().replace({'': 'unknown'})

risk_df = risk_df.merge(
    hotspot_summary[['corridor', 'junction', 'police_station', 'total_events', 'closure_rate', 'hotspot_score', 'hotspot_level']],
    on=['corridor', 'junction', 'police_station'],
    how='left'
)

risk_df[['total_events', 'closure_rate', 'hotspot_score']] = risk_df[['total_events', 'closure_rate', 'hotspot_score']].fillna(0)
risk_df['hotspot_level'] = risk_df['hotspot_level'].fillna('low')

playbooks = risk_df.apply(build_recommendation, axis=1)
playbook_df = pd.json_normalize(playbooks)

risk_playbooks = pd.concat([risk_df.reset_index(drop=True), playbook_df], axis=1)
risk_playbooks['equipment'] = risk_playbooks['equipment'].apply(lambda items: ', '.join(items))
risk_playbooks['agency_alerts'] = risk_playbooks['agency_alerts'].apply(lambda items: ', '.join(items))
risk_playbooks['playbook_json'] = playbooks.apply(json.dumps)

# Put operational columns first.
front_cols = [
    'id', 'event_type', 'event_cause', 'corridor', 'junction', 'police_station', 'zone',
    'road_closure_probability', 'predicted_duration_band', 'prediction_confidence',
    'confidence_bucket', 'risk_level', 'risk_score',
    'manpower', 'barricading', 'diversion', 'equipment', 'agency_alerts',
    'hotspot_level', 'hotspot_score', 'actual_duration_band', 'target_road_closure'
]
front_cols = [c for c in front_cols if c in risk_playbooks.columns]
remaining_cols = [c for c in risk_playbooks.columns if c not in front_cols]
risk_playbooks = risk_playbooks[front_cols + remaining_cols]

print('Risk playbook shape:', risk_playbooks.shape)
print('\nRisk level distribution:')
display(risk_playbooks['risk_level'].value_counts().to_frame('count'))
print('\nConfidence bucket distribution:')
display(risk_playbooks['confidence_bucket'].value_counts().to_frame('count'))
display(risk_playbooks.head(10))

Risk playbook shape: (2764, 35)
Risk level distribution:


,count
risk_level,
medium,1255
low,957
high,372
critical,180


,id,event_type,event_cause,corridor,junction,police_station,zone,road_closure_probability,predicted_duration_band,risk_level,risk_score,manpower,barricading,diversion,equipment,agency_alerts,hotspot_level,hotspot_score,actual_duration_band,target_road_closure,latitude,longitude,start_datetime,description,duration_band,is_peak_hour,valid_duration_label,prob_duration_long,prob_duration_medium,prob_duration_short,total_events,closure_rate,control_room,primary_trigger,playbook_json
0,FKID000001,unplanned,vehicle_breakdown,orr east 1,unknown,hsr layout,unknown,0.479556,short,medium,46.95,2 officers + 1 patrol unit,Local channelization with cones/signage near i...,Prepare local diversion if queue spills to adj...,"clearance crew, reflective jackets, tow van, t...","traffic police, traffic police tow unit",low,14.550000,short,0,12.921876,77.645158,2024-01-30 04:07:24.173000+00:00,Starting problem,short,1,1,0.009359,0.194476,0.796165,20,0.150000,Notify station control and monitor every 15 mi...,vehicle_breakdown,"{""risk_level"": ""medium"", ""risk_score"": 46.95, ..."
1,FKID000004,unplanned,vehicle_breakdown,non-corridor,lalbaghmaingatejunc,wilson garden,unknown,0.692699,short,medium,47.70,2 officers + 1 patrol unit,Local channelization with cones/signage near i...,Prepare local diversion if queue spills to adj...,"clearance crew, reflective jackets, tow van, t...","traffic police, traffic police tow unit",low,1.750000,short,0,12.953980,77.585233,2024-01-30 04:56:32.348000+00:00,[LOCATION] ಪೈಪ್ [PERSON] ವಾಹನ ಆಫ್ ಆಗಿರುತ್ತದೆ ಸರ್,short,0,1,0.000000,0.237178,0.762822,5,0.000000,Notify station control and monitor every 15 mi...,vehicle_breakdown,"{""risk_level"": ""medium"", ""risk_score"": 47.7, ""..."
2,FKID000018,unplanned,tree_fall,non-corridor,unknown,jayanagara,unknown,0.925349,long,critical,100.00,8+ officers + inspector oversight + 3 patrol u...,Full barricade deployment and protected emerge...,Activate full diversion plan and public adviso...,"barricades, portable signage, reflective jacke...","BBMP tree clearance, control room, traffic police",high,43.046154,long,0,12.923352,77.590488,2024-03-07 19:39:29.764000+00:00,ssmrv junction towards 20th main ಬಳಿ ಮರ ಬಿದ್ದಿ...,long,0,1,0.642888,0.208505,0.148607,130,0.107692,Dedicated control-room monitoring until cleara...,tree_fall,"{""risk_level"": ""critical"", ""risk_score"": 100.0..."
3,FKID000020,unplanned,vehicle_breakdown,hosur road,unknown,madiwala,unknown,0.248171,short,medium,33.12,2 officers + 1 patrol unit,Local channelization with cones/signage near i...,Prepare local diversion if queue spills to adj...,"clearance crew, reflective jackets, tow van, t...","traffic police, traffic police tow unit",medium,19.834783,short,0,12.907122,77.628640,2024-03-07 19:25:04.302000+00:00,ಬೊಮ್ಮನಹಳ್ಳಿ ಟವರ್ಡ್ಸ್ ಸಿಟಿ ಟೆಕ್ನಿಕಲ್ ಪ್ರಾಬ್ಲಮ್ ...,short,0,1,0.000000,0.328292,0.671708,46,0.065217,Notify station control and monitor every 15 mi...,vehicle_breakdown,"{""risk_level"": ""medium"", ""risk_score"": 33.12, ..."
4,FKID000025,unplanned,tree_fall,non-corridor,unknown,wilson garden,unknown,0.954939,medium,critical,89.62,8+ officers + inspector oversight + 3 patrol u...,Full barricade deployment and protected emerge...,Activate full diversion plan and public adviso...,"barricades, portable signage, reflective jacke...","BBMP tree clearance, traffic police",medium,28.833333,long,1,12.946391,77.597326,2024-03-07 06:09:19.361000+00:00,no,long,0,1,0.122715,0.504827,0.372458,54,0.185185,Dedicated control-room monitoring until cleara...,tree_fall,"{""risk_level"": ""critical"", ""risk_score"": 89.62..."
5,FKID000028,unplanned,pot_holes,unknown,unknown,hal old airport,unknown,0.276360,short,low,20.54,1 patrol unit,No barricading by default; keep cones ready if...,No diversion. Monitor junction approach roads.,"reflective jackets, traffic cones",traffic police,low,0.350000,short,0,12.963511,77.664212,2024-01-30 08:25:57.212000+00:00,A,short,0,1,0.369342,0.251932,0.378726,1,0.000000,Log and monitor.,default

## 8. Recommendation Summary

In [ ]:
recommendation_summary = pd.DataFrame({
    'metric': [
        'events_scored',
        'low_risk_events',
        'medium_risk_events',
        'high_risk_events',
        'critical_risk_events',
        'avg_road_closure_probability',
        'avg_risk_score',
        'avg_prediction_confidence',
        'high_confidence_pct',
        'medium_confidence_pct',
        'low_confidence_pct',
        'hotspot_rows',
        'cause_overrides_count',
    ],
    'value': [
        len(risk_playbooks),
        int(risk_playbooks['risk_level'].eq('low').sum()),
        int(risk_playbooks['risk_level'].eq('medium').sum()),
        int(risk_playbooks['risk_level'].eq('high').sum()),
        int(risk_playbooks['risk_level'].eq('critical').sum()),
        round(float(risk_playbooks['road_closure_probability'].mean()), 4),
        round(float(risk_playbooks['risk_score'].mean()), 4),
        round(float(risk_playbooks['prediction_confidence'].mean()), 4),
        round(float(risk_playbooks['confidence_bucket'].eq('high').mean() * 100), 2),
        round(float(risk_playbooks['confidence_bucket'].eq('medium').mean() * 100), 2),
        round(float(risk_playbooks['confidence_bucket'].eq('low').mean() * 100), 2),
        len(hotspot_summary),
        len(CAUSE_OVERRIDES),
    ]
})

display(recommendation_summary)

,metric,value
0,events_scored,2764.0000
1,low_risk_events,957.0000
2,medium_risk_events,1255.0000
3,high_risk_events,372.0000
4,critical_risk_events,180.0000
5,avg_road_closure_probability,0.2174
6,avg_risk_score,40.3362
7,hotspot_rows,642.0000


## 9. Save Outputs

In [9]:
risk_playbooks_path = OUTPUT_DIR / 'event_risk_playbooks.csv'
hotspot_summary_path = OUTPUT_DIR / 'hotspot_summary.csv'
rules_path = OUTPUT_DIR / 'recommendation_rules.json'
summary_path = OUTPUT_DIR / 'recommendation_summary.csv'

risk_playbooks.to_csv(risk_playbooks_path, index=False)
hotspot_summary.sort_values('hotspot_score', ascending=False).to_csv(hotspot_summary_path, index=False)
recommendation_summary.to_csv(summary_path, index=False)
with open(rules_path, 'w') as f:
    json.dump(recommendation_rules, f, indent=2)

print('Saved risk playbooks:', risk_playbooks_path)
print('Saved hotspot summary:', hotspot_summary_path)
print('Saved rules:', rules_path)
print('Saved summary:', summary_path)

Saved risk playbooks: /Users/astron_designer/GridLock_Phase2/outputs/recommendations/event_risk_playbooks.csv
Saved hotspot summary: /Users/astron_designer/GridLock_Phase2/outputs/recommendations/hotspot_summary.csv
Saved rules: /Users/astron_designer/GridLock_Phase2/outputs/recommendations/recommendation_rules.json
Saved summary: /Users/astron_designer/GridLock_Phase2/outputs/recommendations/recommendation_summary.csv


## 10. Example Playbook

This cell shows one critical/high risk event as a readable operational playbook.

In [10]:
example = risk_playbooks.sort_values('risk_score', ascending=False).iloc[0]
print('Event ID:', example.get('id', 'unknown'))
print('Cause:', example.get('event_cause', 'unknown'))
print('Corridor:', example.get('corridor', 'unknown'))
print('Closure probability:', round(float(example['road_closure_probability']), 3))
print('Predicted duration:', example['predicted_duration_band'])
print('Risk:', example['risk_level'], round(float(example['risk_score']), 2))
print('Manpower:', example['manpower'])
print('Barricading:', example['barricading'])
print('Diversion:', example['diversion'])
print('Equipment:', example['equipment'])
print('Agency alerts:', example['agency_alerts'])

Event ID: FKID003815
Cause: tree_fall
Corridor: non-corridor
Closure probability: 0.939
Predicted duration: long
Risk: critical 100.0
Manpower: 8+ officers + inspector oversight + 3 patrol units
Barricading: Full barricade deployment and protected emergency lane.
Diversion: Activate full diversion plan and public advisory immediately.
Equipment: barricades, portable signage, reflective jackets, traffic cones, tree clearance crew
Agency alerts: BBMP tree clearance, control room, traffic police


## 11. Final Checklist

In [ ]:
assert risk_playbooks_path.exists(), risk_playbooks_path
assert hotspot_summary_path.exists(), hotspot_summary_path
assert rules_path.exists(), rules_path
assert summary_path.exists(), summary_path
assert {'risk_level', 'risk_score', 'manpower', 'diversion', 'confidence_bucket'}.issubset(risk_playbooks.columns)

print('=' * 72)
print('BLOCK 5 COMPLETE: RISK SCORING + RECOMMENDATION ENGINE (IMPROVED)')
print('=' * 72)
print('Events scored:', len(risk_playbooks))
print('Cause overrides:', len(CAUSE_OVERRIDES))
print('Confidence-aware scoring: enabled')
print('Outputs saved under:', OUTPUT_DIR)
print('=' * 72)

BLOCK 5 COMPLETE: RISK SCORING + RECOMMENDATION ENGINE
Events scored: 2764
Outputs saved under: /Users/astron_designer/GridLock_Phase2/outputs/recommendations
